In [15]:
from typing import TypedDict, Literal
from langgraph.graph import START, END, StateGraph
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
class WorkflowState(TypedDict):
    query: str
    category: str
    response: str


class CategoryClassification(BaseModel):
    category:Literal["billing", "technical"] = Field(
        description="Classify the intent into 'billing' (payments, refunds, pricing) or  'technical' (bugs, uptime, API errors, accounts)."
    )

In [17]:
def classifier_node(state:WorkflowState):
    print("-----[NODE] Classifying Query -----")
    user_query = state['query'].lower()
    llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)
    structured_llm = llm.with_structured_output(CategoryClassification)

    prompt = ChatPromptTemplate.from_messages([
        ("system","You are an automated routing assistant. Analyze the incoming user inquiry and classify its department destination."),
        ("human","{user_query}")
    ])

    classification_chain = prompt | structured_llm
    prediction = classification_chain.invoke({"user_query": user_query})

    return {"category":prediction.category}


def billing_expert_node(state:WorkflowState):
    print("----[NODE] Processing Billing Ticket----")
    return {"response":"Routing your inquiry directly to our Billing and payment operations team."}


def technical_expert_node(state:WorkflowState):
    print("-----[NODE] Processing technical issue Ticket")
    return {"response":"Opening a high priority ticket with engineering to troubleshoot your system."}

def router_logic(state:WorkflowState) -> Literal["to_billing", "to_technical"]:
    if state["category"]=="billing":
        return "to_billing"
    return "to_technical"



In [18]:
builder = StateGraph(WorkflowState)
builder.add_node("classifier", classifier_node)
builder.add_node("billing_processor", billing_expert_node)
builder.add_node("technical_processor", technical_expert_node)

builder.add_edge(START, "classifier")

builder.add_conditional_edges(
    "classifier",
    router_logic,
    {
        "to_billing":"billing_processor",
        "to_technical":"technical_processor"
    }
)

builder.add_edge("billing_processor",END)
builder.add_edge("technical_processor",END)

graph = builder.compile()

In [19]:
input_state_1 = {"query":"I need help with my monthly invoice payment."}
result_1 = graph.invoke(input_state_1)
print(f"Result 1 category:{result_1['category']}")
print(f"Result 1 response:{result_1['response']}")

-----[NODE] Classifying Query -----
----[NODE] Processing Billing Ticket----
Result 1 category:billing
Result 1 response:Routing your inquiry directly to our Billing and payment operations team.


In [20]:
input_state_2 = {"query":"The API keeps returning the 500 error code."}
result_2 = graph.invoke(input_state_2)
print(f"Result 2 category:{result_2['category']}")
print(f"Result 2 response:{result_2['response']}")

-----[NODE] Classifying Query -----
-----[NODE] Processing technical issue Ticket
Result 2 category:technical
Result 2 response:Opening a high priority ticket with engineering to troubleshoot your system.
